In [1]:
%%capture
from pathlib import Path

if Path.cwd().stem == "notebooks":
    %cd ..
    %load_ext autoreload
    %autoreload 2

In [34]:
import os
from pathlib import Path

import altair as alt
import holoviews as hv
import numpy as np
import polars as pl
import tomllib
from dotenv import load_dotenv
from polars import col

from src.data.database_manager import DatabaseManager
from src.experiments.measurement.stimulus_generator import StimulusGenerator
from src.features.scaling import scale_min_max
from src.features.utils import to_describe
from src.log_config import configure_logging
from src.plots.sci_data import (
    plot_stimulus_seed_grid,
    plot_stimulus_with_labels,
    plot_stimulus_with_physiological_signals,
)
from src.plots.utils import calculate_z_score

load_dotenv()
FIGURE_DIR = Path(os.getenv("SCI_DATA_FIGURE_DIR"))

configure_logging(
    ignore_libs=("Comm", "bokeh", "tornado", "matplotlib"),
)

pl.Config.set_tbl_rows(12)  # for 12 seeds
hv.output(widget_location="bottom", size=150)

In [35]:
db = DatabaseManager()

In [36]:
with db:
    df = db.get_trials("Explore_Data", exclude_problematic=True)

df = df.rename({"rating": "pain_rating", "pupil": "pupil_diameter"})

df = scale_min_max(
    df,
    exclude_additional_columns=[
        # already normalized:
        "temperature",
        "pain_rating",
        "brow_furrow",
        "cheek_raise",
        "mouth_open",
        "upper_lip_raise",
        "nose_wrinkle",
    ],
)

In [37]:
width = 800
height = 400

## Curve with physiological signals and confidence intervals

In [38]:
signals = [
    "temperature",
    "pain_rating",
    "pupil_diameter",
    "eda_tonic",
    "eda_phasic",
    "heart_rate",
    "mouth_open",
]

confidence_level = 0.95
z_score = calculate_z_score(confidence_level)

df = df.filter(col("stimulus_seed") == 133)

# Group by stimulus seed and normalized timestamp, then calculate mean, std, sem, ci
ci_values = (
    df.group_by(col("normalized_timestamp"), maintain_order=True)
    .agg(
        *[col(c).mean().alias(f"mean_{c}") for c in signals],
        *[col(c).std().alias(f"std_{c}") for c in signals],
        pl.len().alias("n"),
    )
    .sort("normalized_timestamp")
    .with_columns(
        *[(col(f"std_{c}") / col("n").sqrt()).alias(f"sem_{c}") for c in signals],
    )
    .with_columns(
        *[
            (col(f"mean_{c}") - z_score * col(f"sem_{c}")).alias(f"ci_lower_{c}")
            for c in signals
        ],
        *[
            (col(f"mean_{c}") + z_score * col(f"sem_{c}")).alias(f"ci_upper_{c}")
            for c in signals
        ],
    )
)

In [39]:
ci_values

normalized_timestamp,mean_temperature,mean_pain_rating,mean_pupil_diameter,mean_eda_tonic,mean_eda_phasic,mean_heart_rate,mean_mouth_open,std_temperature,std_pain_rating,std_pupil_diameter,std_eda_tonic,std_eda_phasic,std_heart_rate,std_mouth_open,n,sem_temperature,sem_pain_rating,sem_pupil_diameter,sem_eda_tonic,sem_eda_phasic,sem_heart_rate,sem_mouth_open,ci_lower_temperature,ci_lower_pain_rating,ci_lower_pupil_diameter,ci_lower_eda_tonic,ci_lower_eda_phasic,ci_lower_heart_rate,ci_lower_mouth_open,ci_upper_temperature,ci_upper_pain_rating,ci_upper_pupil_diameter,ci_upper_eda_tonic,ci_upper_eda_phasic,ci_upper_heart_rate,ci_upper_mouth_open
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,0.0,0.36777,0.817986,0.842322,0.425671,0.694208,0.063967,0.0,0.245469,0.168467,0.172791,0.294998,0.200352,0.150445,37,0.0,0.040355,0.027696,0.028407,0.048497,0.032938,0.024733,0.0,0.288676,0.763703,0.786646,0.330618,0.629651,0.015491,0.0,0.446864,0.872268,0.897999,0.520724,0.758764,0.112443
100.0,0.000184,0.380709,0.815849,0.843847,0.437848,0.708431,0.064199,0.000025,0.247465,0.178117,0.17077,0.28961,0.202023,0.151244,37,0.000004,0.040683,0.029282,0.028074,0.047612,0.033212,0.024864,0.000176,0.300972,0.758457,0.788822,0.344531,0.643336,0.015466,0.000192,0.460446,0.873242,0.898871,0.531165,0.773526,0.112933
200.0,0.000784,0.400342,0.81457,0.845285,0.455036,0.719241,0.064719,0.000087,0.245996,0.18692,0.168796,0.289537,0.204379,0.152755,37,0.000014,0.040441,0.030729,0.02775,0.0476,0.0336,0.025113,0.000756,0.321078,0.754342,0.790896,0.361742,0.653387,0.015499,0.000812,0.479606,0.874799,0.899673,0.548329,0.785095,0.113939
300.0,0.001871,0.419517,0.811261,0.846712,0.473843,0.727623,0.065076,0.000143,0.246142,0.196225,0.166743,0.290675,0.205764,0.153793,37,0.000023,0.040465,0.032259,0.027412,0.047787,0.033827,0.025283,0.001825,0.340207,0.748034,0.792985,0.380183,0.661322,0.015521,0.001917,0.498828,0.874488,0.900439,0.567504,0.793923,0.114631
400.0,0.003444,0.43083,0.805903,0.848108,0.497161,0.734009,0.065487,0.000197,0.248252,0.205165,0.164857,0.29521,0.206642,0.154679,37,0.000032,0.040812,0.033729,0.027102,0.048532,0.033972,0.025429,0.003381,0.350839,0.739796,0.794989,0.402039,0.667426,0.015647,0.003508,0.510821,0.872011,0.901228,0.592282,0.800592,0.115327
500.0,0.005515,0.443373,0.802354,0.849482,0.521489,0.737206,0.065674,0.00028,0.247539,0.212504,0.162865,0.298543,0.206851,0.155343,37,0.000046,0.040695,0.034935,0.026775,0.04908,0.034006,0.025538,0.005425,0.363612,0.733882,0.797004,0.425293,0.670555,0.01562,0.005605,0.523135,0.870826,0.90196,0.617684,0.803857,0.115728
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
179500.0,0.158899,0.484589,0.355889,0.185851,0.442689,0.529453,0.047111,0.000233,0.248002,0.160824,0.225302,0.145671,0.14707,0.103455,37,0.000038,0.040771,0.026439,0.037039,0.023948,0.024178,0.017008,0.158824,0.404679,0.304069,0.113255,0.395751,0.482065,0.013776,0.158974,0.564499,0.407709,0.258447,0.489626,0.576842,0.080445
179600.0,0.157452,0.471173,0.355701,0.18685,0.438051,0.529808,0.047258,0.000183,0.252247,0.160566,0.226321,0.146786,0.146094,0.103893,37,0.00003,0.041469,0.026397,0.037207,0.024131,0.024018,0.01708,0.157393,0.389895,0.303964,0.113926,0.390755,0.482735,0.013782,0.157511,0.552451,0.407438,0.259775,0.485348,0.576882,0.080734


In [42]:
phy = plot_stimulus_with_physiological_signals(
    ci_values,
    signals,
    width=width,
    height=height,
)
phy.save(FIGURE_DIR / "stimulus_with_physiological_signals_ci.svg")
phy

alt.LayerChart(...)

## Curve with temperature intervals

In [56]:
def load_configuration(file_path: str) -> dict:
    """Load configuration from a TOML file."""
    file_path = Path(file_path)
    with open(file_path, "rb") as file:
        return tomllib.load(file)


config = load_configuration("src/experiments/measurement/measurement_config.toml")[
    "stimulus"
]

dummy_participant = {
    "temperature_baseline": 44.0,
    "temperature_range": 4,  # VAS 0 - VAS 70
}
config.update(dummy_participant)

stimulus = StimulusGenerator(config, seed=133)


In [55]:
lab = plot_stimulus_with_labels(
    stimulus,
    width=width,
    height=height,
)
lab.save(FIGURE_DIR / "stimulus_with_labels_ci.svg")
lab

alt.LayerChart(...)

## All curves 


In [11]:
config["sample_rate"] = 2

stimuli = pl.concat(
    [
        pl.DataFrame(
            {
                "y": StimulusGenerator(config, seed).y,
                "time": np.arange(len(StimulusGenerator(config, seed).y)),
                "seed": np.array([seed] * len(StimulusGenerator(config, seed).y)),
            }
        )
        for seed in config["seeds"]
    ]
)
stimuli.group_by("seed", maintain_order=True).agg(to_describe("y"))

seed,y_count,y_null_count,y_mean,y_std,y_min,y_25%,y_50%,y_75%,y_max
i64,u32,u32,f64,f64,f64,f64,f64,f64,f64
133,360,0,43.784518,1.145271,42.0,42.979949,43.455472,44.640197,45.990678
243,360,0,43.835698,1.130401,42.0,43.001099,43.8487,44.741317,45.969692
265,360,0,43.858845,1.183784,42.0,42.905532,43.851603,44.774859,45.984994
396,360,0,43.955609,1.107732,42.0,43.269657,43.88023,44.825056,45.951559
467,360,0,43.900993,1.133494,42.0,43.147233,43.829296,44.782912,45.966138
658,360,0,43.90003,1.114379,42.0,42.969628,43.911083,44.71232,45.878865
681,360,0,43.794,1.106165,42.0,42.962248,43.69888,44.653396,45.972101
743,360,0,43.912674,1.106842,42.0,43.222426,43.832328,44.77548,45.969461
806,360,0,43.761186,1.191864,42.0,42.743569,43.757714,44.662761,45.998058


In [12]:
# fig.savefig(FIGURE_DIR / "stimulus_seeds.png", dpi=300)

In [45]:
chart = plot_stimulus_seed_grid(stimuli, columns=3, width=260, height=150)
chart = chart.configure_header(labelFontSize=15)
chart.save(FIGURE_DIR / "stimulus_seed_grid.svg")
chart

alt.FacetChart(...)